# 模型權重分布分析：從 FP32 到 4-bit 量化的可視化

## 學習目標

完成本 notebook 後，你將能夠：

1. 理解 LLM 權重的統計分布特性（為何接近常態分布）
2. 視覺化比較 FP32、BF16 與 4-bit NF4 量化後的權重分布差異
3. 理解 NF4（Normal Float 4）量化格式如何針對常態分布設計最佳量化點位
4. 評估量化前後的數值精度損失（量化誤差分布）

## 前置知識

- 熟悉 PyTorch tensor 基本操作
- 了解浮點數精度概念（FP32 / BF16 / INT8）
- 建議先閱讀：`04-kbits-tuning/04-1bits_training/README.md`

## 與相鄰 Notebook 的銜接

- 上一個：`../04-1bits_training/` — 介紹 1-bit 量化基本概念
- 本 notebook：視覺化分析 FP32 vs BF16 vs NF4 的權重分布
- 下一個：`../04-8bits_training/` — 實際使用 `BitsAndBytesConfig` 進行 8-bit 與 4-bit 量化訓練

## 為什麼先看分布？

量化的核心問題是：**如何用最少的 bit 數，最忠實地表示原始數值？** 答案取決於數值的分布形狀。LLM 的權重幾乎都服從接近零均值的常態分布，這個事實直接啟發了 NF4 的設計。本 notebook 用視覺化來建立這個直覺。

In [ ]:
# ============================================================
# 版本鎖定（2026 統一慣例）
# 確保在正確的套件版本下執行
# ============================================================
# pip install \
#   transformers>=4.46 \
#   datasets>=3.0 \
#   bitsandbytes>=0.44 \
#   accelerate>=1.0 \
#   safetensors>=0.4 \
#   torch>=2.4 \
#   matplotlib>=3.9

import transformers
import torch

print(f"transformers: {transformers.__version__}")
print(f"torch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 載入模型

### 2026 標準寫法

```python
model = AutoModelForCausalLM.from_pretrained(
    model_id,                          # HF Hub model id，跨環境可攜
    device_map="auto",                 # 自動分配 GPU/CPU/disk offload
    torch_dtype=torch.bfloat16,        # BF16：比 FP16 更穩定，比 FP32 省一半記憶體
    use_safetensors=True,              # 安全格式，避免 pickle 任意代碼執行
)
```

### 為什麼選 BF16？

| 格式 | 指數位元 | 尾數位元 | 動態範圍 | 精度 | 適用場景 |
|------|----------|----------|----------|------|----------|
| FP32 | 8 | 23 | 極大 | 高 | 訓練參考值、數值敏感計算 |
| FP16 | 5 | 10 | **窄**，易溢位 | 中 | 舊 GPU（Volta 前） |
| BF16 | 8 | 7 | 與 FP32 相同 | 較低 | **2026 推薦**，Ampere+ GPU 硬體支援 |

BF16 共享 FP32 的指數位元數，因此**不會溢位**（FP16 在梯度爆炸時常遇到 `inf`/`nan`），是現代訓練與推論的首選格式。

### device_map='auto' 的語意

`device_map='auto'` 讓 `accelerate` 依照可用記憶體自動分配層到 GPU → CPU → disk，做到 CPU/disk offload。單張 GPU 裝不下時不需要手動拆層。

### safetensors 相對 pickle 的優勢

- **安全**：純張量格式，無法夾帶 Python pickle 中的任意代碼
- **速度**：記憶體映射（mmap），大模型首次載入約快 2-3 倍
- **2026 現況**：HuggingFace Hub 所有主流模型都已提供 `.safetensors` 版本

In [ ]:
from transformers import AutoModelForCausalLM
import torch

# 2026 寫法：使用 HF Hub model id，移除硬路徑
# 選用 Qwen2.5-1.5B-Instruct 作為輕量替代示範
# - 原 notebook 用 ChatGLM3-6B，需約 12 GB VRAM (FP32) / 6 GB (BF16)
# - 改用 1.5B 模型約需 3 GB VRAM (BF16)，便於在教學環境執行
# - 換成較大模型只需改 MODEL_ID 一行
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # ~3 GB VRAM (BF16)
# MODEL_ID = "THUDM/chatglm3-6b-base"     # 若要還原原版，取消此行並刪上行

print(f"Loading model: {MODEL_ID}")
print("VRAM hint: BF16 precision, ~3 GB for 1.5B / ~12 GB for 6B")

# 2026 統一載入慣例
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",          # 自動 GPU/CPU offload
    torch_dtype=torch.bfloat16, # BF16：穩定、省記憶體
    use_safetensors=True,       # 安全格式
    # trust_remote_code 僅在模型確實需要時才開啟
    # ChatGLM 系列需要；Qwen 等主流模型已無需
)
model.eval()  # 純推論 / 分析模式，關閉 dropout

# 確認模型分配情況
print("\nModel device map:")
if hasattr(model, 'hf_device_map'):
    for layer, device in list(model.hf_device_map.items())[:5]:
        print(f"  {layer}: {device}")
    print("  ...")
print(f"\nParameter dtype: {next(model.parameters()).dtype}")

## 2. 提取模型權重

我們要分析的是模型所有可訓練參數（`model.parameters()`）的數值分布。

注意事項：
- 權重載入為 BF16，繪圖前先轉換回 FP32 以確保 `torch.histogram` 的數值精度
- 只取 `requires_grad=False` 的參數也沒關係，本 notebook 目的是分析分布，不是訓練
- 大模型（6B+）的參數量超過 60 億個，全部展開需要數 GB RAM；若記憶體不足可改用抽樣版本

In [ ]:
def get_weights_flat(
    model: torch.nn.Module,
    max_params: int = 50_000_000,  # 預設最多取 5000 萬個數值，避免 OOM
    seed: int = 42,
) -> torch.Tensor:
    """
    Extract a flat tensor of model weight values for distribution analysis.

    Args:
        model: The model to analyze.
        max_params: Maximum number of scalar values to collect.
                    Larger models are randomly subsampled to this limit.
        seed: Random seed for reproducible subsampling.

    Returns:
        1-D float32 tensor of weight values.
    """
    torch.manual_seed(seed)
    collected = []
    total = 0

    for param in model.parameters():
        # 轉成 FP32 再展平，確保後續直方圖計算精確
        flat = param.detach().float().view(-1)
        total += flat.numel()
        collected.append(flat.cpu())  # 移到 CPU 以節省 GPU 顯存

    weights_all = torch.cat(collected)

    if weights_all.numel() > max_params:
        # 隨機抽樣，保持分布代表性
        indices = torch.randperm(weights_all.numel())[:max_params]
        weights_all = weights_all[indices]
        print(f"Total params: {total:,} -> subsampled to {max_params:,}")
    else:
        print(f"Total params: {total:,} (no subsampling needed)")

    return weights_all


weights_fp32 = get_weights_flat(model)
print(f"Weight tensor shape: {weights_fp32.shape}")
print(f"dtype: {weights_fp32.dtype}")

## 3. 基本統計量

在繪圖前先看數字，養成先確認統計量再可視化的習慣。

In [ ]:
import numpy as np

# 基本統計量
mean_val  = weights_fp32.mean().item()
std_val   = weights_fp32.std().item()
min_val   = weights_fp32.min().item()
max_val   = weights_fp32.max().item()
p1, p99   = torch.quantile(weights_fp32, torch.tensor([0.01, 0.99])).tolist()

print("Weight distribution statistics (FP32):")
print(f"  mean  : {mean_val:.6f}")
print(f"  std   : {std_val:.6f}")
print(f"  min   : {min_val:.6f}")
print(f"  max   : {max_val:.6f}")
print(f"  p1    : {p1:.6f}")
print(f"  p99   : {p99:.6f}")
print()
print("Observation: LLM weights are tightly centered near zero,")
print("following an approximately normal (Gaussian) distribution.")
print("This is the key insight behind NF4 quantization design.")

## 4. FP32 權重分布直方圖

這是最基礎的可視化：原始精度下，模型的所有參數值是如何分布的？

觀察重點：
- 分布是否接近鐘形曲線（常態分布）？
- 極端值（outlier）有多少？
- 範圍大約在哪個區間？

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 使用 2 倍標準差作為直方圖範圍，覆蓋約 95% 的數值
# 排除極少數 outlier，讓主體分布更清晰可見
HIST_RANGE = (-2 * std_val, 2 * std_val)
BINS = 200

hist_fp32 = torch.histogram(
    weights_fp32,
    bins=BINS,
    range=HIST_RANGE,
)

bin_edges = hist_fp32.bin_edges.numpy()
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
counts = hist_fp32.hist.numpy()

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(
    bin_centers,
    counts,
    width=(bin_edges[1] - bin_edges[0]),
    color="steelblue",
    alpha=0.8,
    label="FP32 weights",
)

# 疊加常態分布曲線作為對照
from scipy.stats import norm
x_fit = np.linspace(HIST_RANGE[0], HIST_RANGE[1], 500)
y_fit = norm.pdf(x_fit, mean_val, std_val)
# 縮放至直方圖面積
y_fit_scaled = y_fit * (counts.sum() * (bin_edges[1] - bin_edges[0]))
ax.plot(x_fit, y_fit_scaled, color="red", linewidth=2, label=f"Normal(μ={mean_val:.4f}, σ={std_val:.4f})")

ax.set_xlabel("Weight Value", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(f"FP32 Weight Distribution — {MODEL_ID.split('/')[-1]}", fontsize=14)
ax.xaxis.set_major_locator(ticker.MaxNLocator(10))
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("weight_dist_fp32.png", dpi=150)
plt.show()
print("Saved: weight_dist_fp32.png")

## 5. 理解 NF4：為常態分布量身設計的量化格式

### 量化是什麼？

量化（Quantization）是將高精度數值（如 FP32 的 32 位元）對應到低精度離散格式（如 4-bit 的 16 個等級）的過程。

### 為什麼普通的線性量化（INT4）對 LLM 不夠好？

線性量化將數值範圍均勻切成 16 格：

```
INT4: [-0.1, -0.087, -0.073, ..., 0.087, 0.1]  # 等距
```

問題：LLM 的權重**集中在零附近**，兩端很稀疏。等距劃分會把大量「格子」浪費在幾乎沒有數值的兩端，中間最密集的區域反而格子不夠。

### NF4 的解法：分位數量化

NF4（Normal Float 4）使用**常態分布的分位數**來決定 16 個量化點位：

```python
# 概念示意：NF4 量化點位由常態分布 CDF 的等分位數決定
quantiles = norm.ppf(np.linspace(1/32, 1 - 1/32, 16))
# 結果：點位密集在中間、稀疏在兩端，完全符合 LLM 權重分布
```

這樣中間密集區域獲得更多量化點位，**每一個 bit 都物盡其用**。

### Double Quantization 的成本效益

`bnb_4bit_use_double_quant=True` 對量化的縮放因子（scale factor）再做一次量化：
- 縮放因子本身從 FP32 壓縮到 8-bit
- 每個參數約額外節省 0.37 bits
- 7B 模型約節省 0.4 GB VRAM
- 精度損失：可忽略不計

In [ ]:
# 視覺化 NF4 量化點位 vs 線性 INT4 量化點位
# 展示分位數量化如何更貼合常態分布
from scipy.stats import norm
import matplotlib.pyplot as plt
import numpy as np

# NF4 的 16 個量化點位（由常態分布分位數決定）
# 參考：bitsandbytes 源碼中的 NF4_QUANT_TABLE
nf4_quantiles = norm.ppf(np.linspace(1 / 32, 1 - 1 / 32, 16))
nf4_quantiles = nf4_quantiles / np.max(np.abs(nf4_quantiles))  # 正規化到 [-1, 1]

# 線性 INT4 的 16 個等距量化點位
int4_linear = np.linspace(-1, 1, 16)

# 繪圖：常態分布背景 + 兩種量化點位
x = np.linspace(-3, 3, 1000)
y = norm.pdf(x, 0, 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, points, label, color in [
    (axes[0], int4_linear, "INT4 Linear (uniform)", "tomato"),
    (axes[1], nf4_quantiles, "NF4 (normal quantile)", "steelblue"),
]:
    ax.fill_between(x, y, alpha=0.2, color="gray")
    ax.plot(x, y, color="gray", linewidth=1.5)
    for p in points:
        ax.axvline(p * 3, color=color, linewidth=1.2, alpha=0.8)  # 縮放到顯示範圍
    ax.set_title(label, fontsize=12)
    ax.set_xlabel("Normalized Weight Value")
    ax.set_ylabel("Probability Density")
    ax.set_xlim(-3.2, 3.2)

axes[0].text(0.5, 0.92, "Uniform spacing wastes\nresolution in dense center",
             transform=axes[0].transAxes, ha='center', fontsize=9, color='darkred')
axes[1].text(0.5, 0.92, "Denser near zero = less\nquantization error for LLMs",
             transform=axes[1].transAxes, ha='center', fontsize=9, color='navy')

plt.suptitle("Quantization Point Placement: INT4 vs NF4", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("nf4_vs_int4_quantization_points.png", dpi=150)
plt.show()
print("Saved: nf4_vs_int4_quantization_points.png")

## 6. 模擬 4-bit 量化並分析誤差分布

實際量化需要 `bitsandbytes` 的 CUDA kernel，這裡用純 Python 模擬 NF4 量化的概念流程，讓你看到量化誤差的分布形態。

這不是完全等價的實作（真實 NF4 有額外的 block-wise scaling），但足以展示誤差分布的統計特性。

In [ ]:
def simulate_nf4_quantize(weights: torch.Tensor, block_size: int = 64) -> torch.Tensor:
    """
    Simulate NF4 quantization (concept-level, not bit-exact with bitsandbytes).

    The key steps are:
    1. Block-wise normalization (scale each block to [-1, 1])
    2. Map each value to the nearest NF4 quantization point
    3. Reconstruct (dequantize) back to float32

    Args:
        weights: 1-D float32 tensor of weight values.
        block_size: Number of values per quantization block.

    Returns:
        Dequantized float32 tensor (same shape as input).
    """
    # NF4 quantization table (normalized to [-1, 1])
    nf4_table = norm.ppf(np.linspace(1 / 32, 1 - 1 / 32, 16))
    nf4_table = nf4_table / np.max(np.abs(nf4_table))
    nf4_table_t = torch.tensor(nf4_table, dtype=torch.float32)

    w = weights.clone()
    n = w.numel()
    # Pad to multiple of block_size
    pad = (block_size - n % block_size) % block_size
    if pad > 0:
        w = torch.cat([w, torch.zeros(pad)])

    w_reshaped = w.view(-1, block_size)
    # Block-wise absolute max scaling
    abs_max = w_reshaped.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    w_normalized = w_reshaped / abs_max  # scale to [-1, 1]

    # Map each value to nearest NF4 point
    # Shape: (num_blocks, block_size, 16)
    diff = (w_normalized.unsqueeze(-1) - nf4_table_t.unsqueeze(0).unsqueeze(0)).abs()
    nearest_idx = diff.argmin(dim=-1)
    w_quantized = nf4_table_t[nearest_idx]  # dequantize

    # Rescale back
    w_dequantized = w_quantized * abs_max
    # Remove padding and return flat
    return w_dequantized.view(-1)[:n]


print("Simulating NF4 quantization (concept demo, not bit-exact)...")
# Use a subset for speed
subset = weights_fp32[:2_000_000]
weights_nf4_dequant = simulate_nf4_quantize(subset)
quant_error = (subset - weights_nf4_dequant)

print(f"Quantization error stats:")
print(f"  mean  : {quant_error.mean().item():.8f}")
print(f"  std   : {quant_error.std().item():.8f}")
print(f"  max   : {quant_error.abs().max().item():.8f}")

## 7. 並排比較：原始分布 vs 量化後分布 vs 量化誤差

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

BINS = 200
RANGE = (-3 * std_val, 3 * std_val)

# --- Panel 1: FP32 original ---
hist_orig = torch.histogram(subset, bins=BINS, range=RANGE)
ax = axes[0]
be = hist_orig.bin_edges.numpy()
bc = (be[:-1] + be[1:]) / 2
ax.bar(bc, hist_orig.hist.numpy(), width=(be[1] - be[0]), color="steelblue", alpha=0.8)
ax.set_title("FP32 Original Weights", fontsize=12)
ax.set_xlabel("Value")
ax.set_ylabel("Count")

# --- Panel 2: NF4 dequantized ---
hist_nf4 = torch.histogram(weights_nf4_dequant, bins=BINS, range=RANGE)
be2 = hist_nf4.bin_edges.numpy()
bc2 = (be2[:-1] + be2[1:]) / 2
ax2 = axes[1]
ax2.bar(bc2, hist_nf4.hist.numpy(), width=(be2[1] - be2[0]), color="darkorange", alpha=0.8)
ax2.set_title("NF4 Dequantized Weights", fontsize=12)
ax2.set_xlabel("Value")
ax2.set_ylabel("Count")

# --- Panel 3: Quantization error ---
err_range = float(quant_error.abs().quantile(0.99).item())
hist_err = torch.histogram(quant_error, bins=BINS, range=(-err_range, err_range))
be3 = hist_err.bin_edges.numpy()
bc3 = (be3[:-1] + be3[1:]) / 2
ax3 = axes[2]
ax3.bar(bc3, hist_err.hist.numpy(), width=(be3[1] - be3[0]), color="seagreen", alpha=0.8)
ax3.set_title("Quantization Error (FP32 - NF4)", fontsize=12)
ax3.set_xlabel("Error Value")
ax3.set_ylabel("Count")

plt.suptitle(
    f"Weight Distribution Analysis — {MODEL_ID.split('/')[-1]}",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.savefig("weight_dist_comparison.png", dpi=150)
plt.show()
print("Saved: weight_dist_comparison.png")
print()
print("Observation:")
print("  Panel 1: FP32 weights follow near-Gaussian distribution, centered at zero.")
print("  Panel 2: NF4 dequantized weights show the 16 discrete quantization levels.")
print("  Panel 3: Quantization error is small and centered near zero (unbiased).")

## 8. 解讀結果與量化格式比較

### 4-bit vs 8-bit vs 16-bit：何時用哪一個？

| 格式 | VRAM 占用 | 精度損失 | 適用場景 |
|------|-----------|----------|----------|
| FP32 | 基準 (1x) | 無 | 研究、數值敏感任務 |
| BF16 | 0.5x | 極小 | **推論與訓練的預設選擇** |
| INT8 | 0.25x | 小 | 推論加速，可接受輕微降分 |
| NF4 (4-bit) | 0.125x | 中小 | 記憶體受限的微調（QLoRA）|
| INT4 (線性) | 0.125x | 中 | 不推薦用於 LLM（對常態分布不友好）|

### nf4 vs fp4

`bitsandbytes` 的 `BitsAndBytesConfig` 提供兩種 4-bit 格式：

- `bnb_4bit_quant_type='nf4'`：**推薦**。分位數量化，針對常態分布最佳化，學術論文（QLoRA 2023）證明效果更好。
- `bnb_4bit_quant_type='fp4'`：浮點 4-bit，量化點位固定，不針對分布最佳化，通常效果略差。

**結論：LLM 微調請一律選 `nf4`。**

### 2026 標準 BitsAndBytesConfig 寫法

```python
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",         # NF4 > FP4 for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,  # 實際矩陣乘法的精度
    bnb_4bit_use_double_quant=True,    # 對縮放因子再量化，省 ~0.4 GB (7B)
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,    # 統一介面
    device_map="auto",
    use_safetensors=True,
)

# PEFT 微調前的必要步驟：
# prepare_model_for_kbit_training() 修正量化模型的梯度流
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
```

`bnb_4bit_compute_dtype` 與 `torch_dtype` 的區別：
- `torch_dtype`：模型載入後的儲存格式（此時已經是 4-bit，這個 kwarg 對量化模型無效）
- `bnb_4bit_compute_dtype`：量化矩陣做乘法時臨時反量化到的精度，**設為 BF16 同時兼顧速度與精度**

In [ ]:
# 展示 2026 標準 BitsAndBytesConfig 載入寫法（需要 GPU 才能實際執行量化）
# 此 cell 為示範，若無 GPU 會跳過量化步驟並給出提示
from transformers import BitsAndBytesConfig
import torch

if not torch.cuda.is_available():
    print("[INFO] No GPU detected — skipping actual 4-bit model load.")
    print("The code below shows the 2026 standard pattern for reference.")
else:
    print("GPU detected — demonstrating BitsAndBytesConfig load...")
    print("VRAM hint: 4-bit 1.5B model needs ~1.5 GB VRAM")

# 2026 標準寫法展示
print()
print("=" * 60)
print("2026 Standard BitsAndBytesConfig pattern:")
print("=" * 60)
print("""
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',               # NF4 > FP4 for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,   # compute in BF16
    bnb_4bit_use_double_quant=True,          # saves ~0.4 GB for 7B
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto',
    use_safetensors=True,
)

# MUST call before applying PEFT/LoRA
model = prepare_model_for_kbit_training(model)
""")

## 小結

本 notebook 帶你建立了量化的核心直覺：

1. **LLM 權重服從近似常態分布**，集中在零附近，這是量化能成功的統計基礎。

2. **NF4 優於線性 INT4**：分位數量化把量化點位集中在數值密集區，讓 4-bit 的 16 個等級物盡其用。

3. **量化誤差小且無偏**：NF4 的誤差分布本身也接近零均值，不會系統性地偏移權重。

4. **2026 標準寫法**：一律使用 `BitsAndBytesConfig`，PEFT 前必須呼叫 `prepare_model_for_kbit_training()`。

5. **格式選擇優先序**：推論/訓練預設用 BF16；記憶體受限的微調用 NF4 4-bit（QLoRA）；4-bit quant type 選 `nf4` 不選 `fp4`。

## 練習

1. 把 `MODEL_ID` 換成另一個你熟悉的模型（如 `meta-llama/Llama-3.2-1B`），觀察權重分布是否仍服從常態分布。

2. 修改 `get_weights_flat()` 讓它只分析特定層（例如只取 `model.model.layers[0]`），比較淺層與深層的分布差異。

3. 將 `simulate_nf4_quantize()` 改成 `simulate_int4_linear_quantize()`（等距 16 點），並把量化誤差疊在同一張圖上比較，驗證 NF4 誤差確實更小。

4. 閱讀 QLoRA 原始論文（Dettmers et al., 2023）的 Section 3，對照本 notebook 的視覺化結果。

## 下一步

前往 `../04-8bits_training/` 或 `../04-4bits_training/`，實際使用 `BitsAndBytesConfig + SFTTrainer` 對模型進行量化微調（QLoRA），把本 notebook 的理論知識用在真實訓練流程上。